# 02 — Lines and Points in PGA2d

This notebook explores the relationship between lines and points in Projective Geometric Algebra. We introduce the **meet** (intersection) and **join** (line through two points) operations, which are fundamental to geometric computations.

## Learning Objectives

- Represent points and lines as multivectors in PGA2d
- Compute line-point incidence using the outer product
- Use the regressive product (meet) for line-line intersection
- Use the outer product (join) for point-point line construction
- Visualize geometric relationships

In [ ]:
# Setup
import matplotlib.pyplot as plt
import numpy as np

from amsa import Algebra

alg = Algebra.pga2d()

## 2.1 Point and Line Representations

In PGA2d:

- **Points** in the join/meet algebra are vectors: $P = w e_0 - y e_1 + x e_2$
  - For finite points: $w \neq 0$ and coordinates come from $x = e_2 / e_0$, $y = -e_1 / e_0$
  - For ideal points: $e_0 = 0$

- **Lines** are bivectors: $L = a e_{01} + b e_{02} + c e_{12}$
  - Represents the Cartesian line $a x + b y + c = 0$

The codebase links these two views through the Poincare complement: `point.poincare_dual()` produces the bivector point encoding used by `amsa.viz`.

In [ ]:
# Create vector-form points in PGA2d
p1 = alg.multivector({"e0": 1.0, "e1": -1.0, "e2": 1.0})  # (1, 1)
p2 = alg.multivector({"e0": 1.0, "e1": -1.0, "e2": 3.0})  # (3, 1)
p3 = alg.multivector({"e0": 1.0, "e1": -3.0, "e2": 2.0})  # (2, 3)

print("Point P1 (1, 1):", p1.values)
print("Point P2 (3, 1):", p2.values)
print("Point P3 (2, 3):", p3.values)

# Coordinate extraction for the vector-form point encoding
print("\nCoordinates:", p1.component("e2") / p1.component("e0"), -p1.component("e1") / p1.component("e0"))
print("Plottable form:", p1.poincare_dual().values)

In [ ]:
# Create lines
# Line: x + y - 2 = 0 → coefficients: a=1, b=1, c=-2
line1 = alg.multivector({"e01": 1.0, "e02": 1.0, "e12": -2.0})

# Line: y - 1 = 0 (horizontal) → a=0, b=1, c=-1
line2 = alg.multivector({"e01": 0.0, "e02": 1.0, "e12": -1.0})

# Line: x = 2 (vertical) → a=1, b=0, c=-2
line3 = alg.multivector({"e01": 1.0, "e02": 0.0, "e12": -2.0})

print("Line L1 (x + y - 2 = 0):", line1.values)
print("Line L2 (y - 1 = 0):", line2.values)
print("Line L3 (x - 2 = 0):", line3.values)

## 2.2 Point-Line Incidence

A point lies on a line when their outer product collapses to zero pseudoscalar:

$$P \wedge L = 0 \Leftrightarrow P \text{ is on } L$$

This is the **incidence test**.

In [ ]:
# Check incidence by plugging vector-form point coordinates into the line equation
def vector_point_xy(point):
    return (
        point.component("e2") / point.component("e0"),
        -point.component("e1") / point.component("e0"),
    )

def line_residual(point, line):
    x, y = vector_point_xy(point)
    return (
        line.component("e01") * x
        + line.component("e02") * y
        + line.component("e12")
    )

r1 = line_residual(p1, line1)
print("P1 residual for L1 =", r1)
print("P1 is on L1:", np.isclose(r1, 0.0))

# Test another point
p_test = alg.multivector({"e0": 1.0, "e1": 0.0, "e2": 2.0})  # (2, 0)
r2 = line_residual(p_test, line1)
print("\nP(2, 0) residual for L1 =", r2)
print("P(2, 0) is on L1:", np.isclose(r2, 0.0))


## 2.3 Meet (Intersection of Lines)

The **meet** of two lines is their intersection point. In PGA, we use the **regressive product** for this:

$$P = L_1 \vee L_2$$

This is computed as: $L_1 \vee L_2 = (L_1 \cdot I^{-1}) \cdot (L_2 \cdot I^{-1})$ where $I$ is the pseudoscalar.

In AMSA, we use `.regress(other)` for the meet operation.

In [ ]:
# Meet of two lines
# L1: y = 1 (horizontal through y=1)
# L2: x = 2 (vertical through x=2)

L1 = alg.multivector({"e01": 0.0, "e02": 1.0, "e12": -1.0})
L2 = alg.multivector({"e01": 1.0, "e02": 0.0, "e12": -2.0})

# Meet: intersection point
intersection = L1.regress(L2)

print("Line 1 (y = 1):", L1.values)
print("Line 2 (x = 2):", L2.values)
print("\nIntersection (meet):", intersection.values)
print("\nVector-form point coordinates: x =", intersection.component("e2") / intersection.component("e0"), ", y =", -intersection.component("e1") / intersection.component("e0"))
intersection_plot = intersection.poincare_dual()
print("Plottable point form:", intersection_plot.values)

In [ ]:
# Visualize the intersection
fig, ax = plt.subplots(figsize=(8, 8))

# Draw L1: y = 1
x_vals = np.linspace(-1, 4, 100)
ax.plot(x_vals, np.ones_like(x_vals), 'b-', linewidth=2, label='L1: y = 1')

# Draw L2: x = 2
ax.plot(np.ones_like(x_vals)*2, x_vals, 'r-', linewidth=2, label='L2: x = 2')

# Draw intersection point
ix = intersection_plot.component("e01") / intersection_plot.component("e12")
iy = intersection_plot.component("e02") / intersection_plot.component("e12")
ax.scatter([ix], [iy], s=200, c='green', zorder=5, marker='*')
ax.text(ix + 0.1, iy + 0.1, f'Intersection ({ix}, {iy})', fontsize=10)

ax.set_xlim(-1, 4)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.set_title('Meet: Line Intersection', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

### Join of two points

In the current AMSA plotting/motor convention, points are often stored in bivector form
`P* = x e01 + y e02 + w e12`. The regressive product of two such points gives a
vector-form line, so we dual that result back into bivector form when we want to read
the familiar coefficients `a, b, c` of `a x + b y + c = 0`.


In [ ]:
# Join: line through two plotting/motor-form points
P1 = alg.multivector({"e01": 1.0, "e02": 1.0, "e12": 1.0})  # (1, 1)
P2 = alg.multivector({"e01": 3.0, "e02": 2.0, "e12": 1.0})  # (3, 2)

# Regressive product returns a vector-form line here
line_vector = P1.regress(P2)
line = line_vector.poincare_dual()

print("Point P1 (1, 1):", P1.values)
print("Point P2 (3, 2):", P2.values)
print("\nVector-form line:", line_vector.values)
print("Bivector-form line:", line.values)

# Extract line coefficients from the bivector-form line
x_coeff = line.component("e01")
y_coeff = line.component("e02")
const_coeff = line.component("e12")
print(f"\nLine equation: {x_coeff}x + {y_coeff}y + {const_coeff} = 0")


In [ ]:
# Verify both points satisfy the extracted Euclidean line equation
def residual_from_plot_point(point, a, b, c):
    x = point.component("e01") / point.component("e12")
    y = point.component("e02") / point.component("e12")
    return a * x + b * y + c

r1 = residual_from_plot_point(P1, x_coeff, y_coeff, const_coeff)
r2 = residual_from_plot_point(P2, x_coeff, y_coeff, const_coeff)

print("P1 on line:", np.isclose(r1, 0.0), "residual=", r1)
print("P2 on line:", np.isclose(r2, 0.0), "residual=", r2)


## 2.5 Visualizing Join

Let's visualize two points and their join (the line through them).

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))

# Points
ax.scatter([1, 3], [1, 2], s=100, c='blue', zorder=5)
ax.text(1.1, 1.1, 'P1 (1,1)', fontsize=10)
ax.text(3.1, 2.1, 'P2 (3,2)', fontsize=10)

# Draw the line through them
a = line.component("e01")
b = line.component("e02")
c = line.component("e12")

x_vals = np.linspace(-1, 5, 100)
y_vals = -(a * x_vals + c) / b
ax.plot(x_vals, y_vals, 'r-', linewidth=2, label=f'Line: {a}x + {b}y + {c:.1f} = 0')

ax.set_xlim(-1, 5)
ax.set_ylim(-1, 4)
ax.set_aspect('equal')
ax.set_title('Join: Line Through Two Points', fontsize=12)
ax.set_xlabel('x')
ax.set_ylabel('y')
ax.grid(True, alpha=0.3)
ax.legend()
plt.show()

## 2.6 Parallel Lines Meet at Infinity

One of the beautiful properties of PGA: parallel lines meet at a **point at infinity**!

When two lines are parallel, their meet has `e0 = 0` in the vector-form point encoding. Its Poincare-dual plotting form then has `e12 = 0`.

In [ ]:
# Two parallel lines
L1 = alg.multivector({"e01": 1.0, "e02": 0.0, "e12": 0.0})   # x = 0
L2 = alg.multivector({"e01": 1.0, "e02": 0.0, "e12": -2.0})  # x = 2

# Meet (intersection)
parallel_meet = L1.regress(L2)

print("Parallel lines: x = 0 and x = 2")
print("Meet:", parallel_meet.values)
print("\ne0 component:", parallel_meet.component("e0"))
print("Plotting form:", parallel_meet.poincare_dual().values)
print("This is a point at infinity (e0 = 0)!" if parallel_meet.component("e0") == 0 else "This is a finite point")

## 2.7 Summary

We covered:

- **Points**: Vector-form elements `w e0 - y e1 + x e2`, with `e0 = 0` for ideal points
- **Lines**: Bivectors `a e01 + b e02 + c e12` representing `a x + b y + c = 0`
- **Incidence test**: `P | L` collapses to zero when the vector-form point lies on the line
- **Meet** (`.regress()`): Intersection of two lines → vector-form point
- **Join in the current AMSA plotting/motor point encoding**: `P1.regress(P2)` gives the line bivector through two bivector-form points
- **Parallel lines**: Meet at ideal vector-form points with `e0 = 0`

In the next notebook, we'll explore **motors** — the PGA representation of rigid body motions.


## Exercises

### ⭐ Easy

**2.1** Create points P1 = (0, 0) and P2 = (1, 1). Find the line through them using the join (outer product). Verify both points lie on this line.

In [ ]:
# Your turn: ⭐ Exercise 2.1
P1 = alg.multivector({"e0": 1.0, "e1": 0.0, "e2": 0.0})
P2 = alg.multivector({"e0": 1.0, "e1": -1.0, "e2": 1.0})
# TODO: Compute join and verify incidence
raise NotImplementedError("Implement exercise 2.1")

### ⭐⭐ Medium

**2.2** Create three lines forming a triangle: L1: x = 0, L2: y = 0, L3: x + y - 1 = 0. Find the three vertices using the meet operation. Verify each vertex lies on two of the three lines.

In [ ]:
# Your turn: ⭐⭐ Exercise 2.2
L1 = alg.multivector({"e01": 1.0, "e02": 0.0, "e12": 0.0})   # x = 0
L2 = alg.multivector({"e01": 0.0, "e02": 1.0, "e12": 0.0})   # y = 0
L3 = alg.multivector({"e01": 1.0, "e02": 1.0, "e12": -1.0}) # x + y - 1 = 0
# TODO: Find vertices using meet
raise NotImplementedError("Implement exercise 2.2")

### ⭐⭐⭐ Challenge

**2.3** Write a function `circle_through_three_points(p1, p2, p3)` that finds the center and radius of a circle passing through three non-collinear points. Use the meet and join operations. Hint: the perpendicular bisectors of the segments meet at the center.

In [ ]:
# Your turn: ⭐⭐⭐ Exercise 2.3
def circle_through_three_points(p1, p2, p3):
    """Find circle through three points. Return (center, radius)."""
    # TODO: Use perpendicular bisectors and meet
    raise NotImplementedError("Implement circle function")

# Test: equilateral triangle
c1 = alg.multivector({"e0": 1.0, "e1": 0.0, "e2": 0.0})
c2 = alg.multivector({"e0": 1.0, "e1": 0.0, "e2": 1.0})
c3 = alg.multivector({"e0": 1.0, "e1": -np.sqrt(3)/2, "e2": 0.5})
center, radius = circle_through_three_points(c1, c2, c3)
print("Center:", center.values if center else "None")
print("Radius:", radius)

## Attribution

This notebook draws on:

- **Projective Geometric Algebra** — Charles G. Gunn
  https://arxiv.org/abs/1901.05873
- **SIGGRAPH 2019 Course Notes** — Charles G. Gunn
  https://arxiv.org/abs/2002.04509
- **PGABLE Tutorial** — Leger and Mann
  https://cs.uwaterloo.ca/~smann/PGABLE/PGAtutorial.pdf
- **Geometric Algebra for Computer Science** — Dorst, Lewiner, et al.
  https://geometricalgebra.org/